In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import islice
from numbers import Number
import pyarrow.parquet as pq

# Configuration
BASE_DIR = Path('multimodal_spectroscopic_dataset')
MAX_POINTS = 2000

if not BASE_DIR.exists():
    raise FileNotFoundError(f'Expected folder not found: {BASE_DIR.resolve()}')
print(f'Inspecting data under: {BASE_DIR.resolve()}')

Inspecting data under: /workspace/gaohanyu/data/raw_data/multimodal_spectroscopic_dataset


In [2]:
total_molecules = 0
file_counts = []

print("Start counting molecules using Parquet metadata...")

# Iterate through files aligned_chunk_0.parquet to aligned_chunk_244.parquet
for i in range(245):
    filename = f'aligned_chunk_{i}.parquet'
    file_path = BASE_DIR / filename

    if not file_path.exists():
        print(f'{filename} not found.')
        continue

    try:
        # Use Parquet metadata to get the row count without materializing the table
        parquet_file = pq.ParquetFile(file_path)
        count = parquet_file.metadata.num_rows
    except Exception as meta_err:
        print(f'Metadata read failed for {filename} ({meta_err}), falling back to pandas.')
        df = pd.read_parquet(file_path)
        count = len(df)

    file_counts.append({
        'file': filename,
        'molecules': count
    })
    total_molecules += count
    print(f'{filename}: {count} molecules')

counts_df = pd.DataFrame(file_counts)
print(f'\nFiles with zero molecules: {counts_df[counts_df["molecules"] == 0]["file"].tolist()}')
print(f'Total molecules: {total_molecules}')
counts_df

Start counting molecules using Parquet metadata...
aligned_chunk_0.parquet: 3202 molecules
aligned_chunk_1.parquet: 3252 molecules
aligned_chunk_2.parquet: 3233 molecules
aligned_chunk_3.parquet: 3278 molecules
aligned_chunk_4.parquet: 3245 molecules
aligned_chunk_5.parquet: 3253 molecules
aligned_chunk_6.parquet: 3258 molecules
aligned_chunk_7.parquet: 3242 molecules
aligned_chunk_8.parquet: 3204 molecules
aligned_chunk_9.parquet: 3203 molecules
aligned_chunk_10.parquet: 3214 molecules
aligned_chunk_11.parquet: 3168 molecules
aligned_chunk_12.parquet: 3287 molecules
aligned_chunk_13.parquet: 3216 molecules
aligned_chunk_14.parquet: 3215 molecules
aligned_chunk_15.parquet: 3192 molecules
aligned_chunk_16.parquet: 3266 molecules
aligned_chunk_17.parquet: 3258 molecules
aligned_chunk_18.parquet: 3243 molecules
aligned_chunk_19.parquet: 3273 molecules
aligned_chunk_20.parquet: 3213 molecules
aligned_chunk_21.parquet: 3244 molecules
aligned_chunk_22.parquet: 3261 molecules
aligned_chunk_23

,file,molecules
0,aligned_chunk_0.parquet,3202
1,aligned_chunk_1.parquet,3252
2,aligned_chunk_2.parquet,3233
3,aligned_chunk_3.parquet,3278
4,aligned_chunk_4.parquet,3245
...,...,...
240,aligned_chunk_240.parquet,3208
241,aligned_chunk_241.parquet,3258
242,aligned_chunk_242.parquet,3237
243,aligned_chunk_243.parquet,3269


In [6]:
# Inspect the first file to see the data structure
first_file = BASE_DIR / 'aligned_chunk_0.parquet'
if first_file.exists():
    df_sample = pd.read_parquet(first_file)
    print("Columns:", df_sample.columns.tolist())
    print("\nData Types:")
    print(df_sample.dtypes)
    print("\nFirst 5 rows:")
    display(df_sample.head())
else:
    print(f"{first_file} does not exist.")

Columns: ['smiles', 'hsqc_nmr_peaks', 'hsqc_nmr_spectrum', 'h_nmr_peaks', 'h_nmr_spectra', 'molecular_formula', 'c_nmr_peaks', 'ir_spectra', 'msms_cfmid_positive_10ev', 'msms_cfmid_positive_20ev', 'msms_cfmid_positive_40ev', 'msms_cfmid_fragments_positive', 'msms_cfmid_negative_10ev', 'msms_cfmid_negative_20ev', 'msms_cfmid_negative_40ev', 'msms_cfmid_fragments_negative', 'c_nmr_spectra', 'msms_iceberg_positive', 'msms_iceberg_fragments_positive', 'msms_scarf_positive', 'msms_scarf_fragments_positive']

Data Types:
smiles                             object
hsqc_nmr_peaks                     object
hsqc_nmr_spectrum                  object
h_nmr_peaks                        object
h_nmr_spectra                      object
molecular_formula                  object
c_nmr_peaks                        object
ir_spectra                         object
msms_cfmid_positive_10ev           object
msms_cfmid_positive_20ev           object
msms_cfmid_positive_40ev           object
msms_cfmid_fragme

,smiles,hsqc_nmr_peaks,hsqc_nmr_spectrum,h_nmr_peaks,h_nmr_spectra,molecular_formula,c_nmr_peaks,ir_spectra,msms_cfmid_positive_10ev,msms_cfmid_positive_20ev,...,msms_cfmid_fragments_positive,msms_cfmid_negative_10ev,msms_cfmid_negative_20ev,msms_cfmid_negative_40ev,msms_cfmid_fragments_negative,c_nmr_spectra,msms_iceberg_positive,msms_iceberg_fragments_positive,msms_scarf_positive,msms_scarf_fragments_positive
0,COc1nc2ccccc2cc1C(=O)O,"[{'13C_centroid': 53.73611104165938, '13C_max'...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[{'category': 'd', 'centroid': 8.5969839833745...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",C11H9NO3,"[{'delta (ppm)': 167.96586334436208, 'integral...","[-0.004481, 0.007279, -0.006439, 0.007464, -0....","[[186.05495, 76.15], [204.06552, 100.0]]","[[158.06004, 32.71], [160.07569, 64.49], [186....",...,"[[101.03858, [C+]#CC1=CC=CC=C1], [103.05423, C...","[[158.06114, 33.71], [202.05097, 100.0]]","[[158.06114, 100.0], [202.05097, 10.6]]","[[128.05057, 81.23], [158.06114, 100.0]]","[[128.05057, [C-]1=NC2=CC=CC=C2C=C1], [158.061...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[25.007277, 0.1479929], [26.015102, 0.1153659...","[[100.015495, C4O3H4], [100.01817, C7NH4], [10...","[[38.01565, 0.28428224], [39.023476, 0.2683198...","[[100.01872, C7NH2], [100.0313, C8H4], [101.00..."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[{'13C_centroid': 27.85448722115822, '13C_max'...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[{'category': 'm', 'centroid': 7.7694256285600...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",C25H24FNO6,"[{'delta (ppm)': 162.9356568279513, 'integral'...","[0.028195, 0.011433, 0.024865, -0.00816, 0.055...","[[149.03972, 100.0], [151.05537, 69.88], [398....","[[57.06988, 6.04], [149.03972, 100.0], [151.05...",...,"[[121.0448, C#CC1=CC=C([FH+])C=C1], [133.06479...","[[135.02517, 9.56], [364.09906, 13.41], [378.0...","[[18.99895, 8.33], [135.02517, 100.0], [149.04...","[[41.00329, 2.71], [41.99854, 5.63], [45.03459...","[[116.0717, CC(C)(C)OC(=[N-])O], [118.08735, C...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[29.038576, 0.26877016], [30.046402, 0.204256...","[[101.05971, C5O2H9], [102.06753, C5O2H9], [10...","[[90.04695, 19.681221], [102.04695, 14.094678]...","[[102.04695, C8H6], [104.026215, C7OH4], [104...."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[{'13C_centroid': 13.692843998619852, '13C_max...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[{'category': 'm', 'centroid': 7.8561744822557...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",C26H25BrN2O4,"[{'delta (ppm)': 168.43702792894814, 'integral...","[-0.003043, 0.014693, -0.003067, 0.025262, -0....","[[162.05495, 8.65], [509.10705, 100.0]]","[[160.0393, 41.0], [162.05495, 29.17], [273.98...",...,"[[105.03349, [O+]#CC1=CC=CC=C1], [117.03349, [...","[[146.02475, 5.59], [507.09249, 100.0]]","[[78.91889, 13.28], [146.02475, 26.61], [160.0...","[[41.99854, 14.3], [77.03967, 5.13], [78.91889...","[[146.02475, C=C=C1C(=O)N=C([O-])C1=C=C], [160...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[28.030752, 0.16267453], [29.002192, 0.216661...","[[103.01784, C7OH4], [104.025665, C7OH4], [105...","[[102.04695, 0.5372584], [104.0626, 0.80846834...","[[102.04695, C8H6], [104.0626, C8H8], [114.046..."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[{'13C_centroid': 14.18117652353497, '13C_max'...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[{'category': 'd', 'centroid': 4.6380422970919...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",C10H21NO5,"[{'delta (ppm)': 103.32093780979817, 'integral...","[0.099321, 0.155452, 0.048156, 0.19118, 0.0237...","[[86.09643, 8.41], [218.13868, 10.46], [236.14...","[[44.04948, 11.53], [56.04948, 5.02], [57.0334...",...,"[[100.11208, C=CCC[NH2+]CC], [110.09643, C#CC=...","[[114.09244, 16.9], [216.12413, 9.97], [234.13...","[[41.00329, 9.32], [59.01385, 42.34], [75.0087...","[[40.01927, 5.88], [41.00329

In [3]:
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from rdkit import DataStructs
    print("RDKit is available.")
except ImportError:
    print("RDKit is NOT available. Please install it.")

RDKit is available.


In [4]:
# Load all files to process the full dataset
from tqdm import tqdm
import concurrent.futures

# Range is 0 to 244
all_files = [BASE_DIR / f'aligned_chunk_{i}.parquet' for i in range(245)]
dfs = []
print("Loading all data files (SMILES only for speed)...")

def load_file(path):
    if path.exists():
        # Optimization: Only load 'smiles' column. 
        # The other columns (spectra etc.) are very large and not needed for fingerprinting/tokenization.
        return pd.read_parquet(path, columns=['smiles'])
    return None

# Use ThreadPoolExecutor for parallel reading
with concurrent.futures.ThreadPoolExecutor() as executor:
    # map returns an iterator, convert to list to trigger execution and show progress
    results = list(tqdm(executor.map(load_file, all_files), total=len(all_files)))

dfs = [r for r in results if r is not None]

combined_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(combined_df)} molecules from all files.")
print("Columns:", combined_df.columns.tolist())

Loading all data files (SMILES only for speed)...


100%|██████████| 245/245 [00:00<00:00, 766.28it/s]

Loaded 794386 molecules from all files.
Columns: ['smiles']


In [6]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
import numpy as np
import faiss
import time
import json
import pickle
from tqdm import tqdm
import concurrent.futures
import os

# 1. Prepare Data
# We use the already loaded combined_df
smiles = combined_df['smiles'].tolist()
# Use dataframe index as IDs
ids = combined_df.index.tolist()

print(f"Total molecules loaded: {len(smiles)}")

# 2. Compute Fingerprints (Parallelized)
def process_chunk(args):
    start_idx, chunk_smiles = args
    # Re-initialize generator in each process
    mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    
    chunk_results = []
    for i, s in enumerate(chunk_smiles):
        try:
            m = Chem.MolFromSmiles(s)
            if m is not None:
                fp = mfgen.GetFingerprint(m)
                chunk_results.append((start_idx + i, fp))
        except:
            continue
    return chunk_results

print(f"Computing fingerprints for {len(smiles)} molecules...")
start_time = time.time()

# Split into chunks
chunk_size = 10000
chunks = []
for i in range(0, len(smiles), chunk_size):
    chunks.append((i, smiles[i:i+chunk_size]))

valid_indices = []
fps = []

# Use ProcessPoolExecutor for CPU-bound task (Fingerprinting is pure Python/RDKit, safe for ProcessPool)
with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_chunk, chunks), total=len(chunks), desc="Computing FPs"))

# Flatten results
for chunk_res in results:
    for idx, fp in chunk_res:
        valid_indices.append(idx)
        fps.append(fp)

print(f"Computed {len(fps)} fingerprints in {time.time() - start_time:.2f} seconds.")

# Map valid index back to original ID
valid_ids = [ids[i] for i in valid_indices]

# 3. Convert to Packed Numpy for Faiss (Parallelized)
def convert_chunk_to_packed(chunk_fps):
    n = len(chunk_fps)
    nbits = 2048
    arr = np.zeros((n, nbits), dtype=np.uint8)
    for i, fp in enumerate(chunk_fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return np.packbits(arr, axis=1)

print("Converting fingerprints to packed numpy for Faiss...")
# Split fps into chunks
fps_chunks = [fps[i:i+chunk_size] for i in range(0, len(fps), chunk_size)]

packed_chunks = []
with concurrent.futures.ProcessPoolExecutor() as executor:
    packed_chunks = list(tqdm(executor.map(convert_chunk_to_packed, fps_chunks), total=len(fps_chunks), desc="Packing FPs"))

packed_fps = np.vstack(packed_chunks)
print(f"Packed fingerprints shape: {packed_fps.shape}")

# 4. Build Faiss Index
d = 2048
print("Building Faiss IndexBinaryFlat...")
# Set Faiss to single thread mode to avoid conflicts with our parallel executor
faiss.omp_set_num_threads(1)
index = faiss.IndexBinaryFlat(d)
index.add(packed_fps)
print(f"Faiss index built with {index.ntotal} vectors.")

# 5. Search and Re-rank (Parallelized)
k_search = 100 # Retrieve more for re-ranking
k_final = 20
batch_size = 1000

# Define function for parallel execution
def search_and_rerank_batch(batch_indices):
    start, end = batch_indices
    query_batch = packed_fps[start:end]
    
    # Faiss Search
    # index.search releases GIL
    D, I = index.search(query_batch, k_search)
    
    batch_results = {}
    
    for i in range(len(query_batch)):
        query_idx = start + i
        query_id = valid_ids[query_idx]
        query_fp = fps[query_idx]
        
        candidates_indices = I[i]
        
        # Filter valid candidates
        valid_candidates_indices = [c for c in candidates_indices if c != -1]
        candidates_fps = [fps[c] for c in valid_candidates_indices]
        candidates_ids = [valid_ids[c] for c in valid_candidates_indices]
        
        # Compute exact Tanimoto
        # BulkTanimotoSimilarity releases GIL
        sims = DataStructs.BulkTanimotoSimilarity(query_fp, candidates_fps)
        
        # Pair (id, similarity)
        candidates_with_sim = list(zip(candidates_ids, sims))
        
        # Sort by similarity descending
        candidates_with_sim.sort(key=lambda x: x[1], reverse=True)
        
        # Filter out self and duplicates
        final_neighbors = []
        seen_neighbors = set()
        
        for cid, sim in candidates_with_sim:
            if cid != query_id and cid not in seen_neighbors:
                final_neighbors.append(cid)
                seen_neighbors.add(cid)
            if len(final_neighbors) >= k_final:
                break
        
        batch_results[query_id] = final_neighbors
    return batch_results

print("Searching and re-ranking (Parallelized)...")
# Create batches
batches = []
for b in range(0, len(packed_fps), batch_size):
    end = min(b + batch_size, len(packed_fps))
    batches.append((b, end))

similarity_map = {}

# Use ThreadPoolExecutor instead of ProcessPoolExecutor for the search part.
# ProcessPoolExecutor can cause deadlocks with Faiss/OpenMP in Jupyter environments.
# ThreadPoolExecutor is safe here because Faiss and RDKit release the GIL for heavy ops.
with concurrent.futures.ThreadPoolExecutor() as executor:
    results = list(tqdm(executor.map(search_and_rerank_batch, batches), total=len(batches), desc="Parallel Search"))

for res in results:
    similarity_map.update(res)

# 6. Save Results
output_file = 'similarity_index.json'
with open(output_file, 'w') as f:
    json.dump(similarity_map, f)

print(f"Saved similarity index to {output_file}")

# Show a sample
if len(similarity_map) > 0:
    sample_id = list(similarity_map.keys())[0]
    print(f"Sample Molecule ID: {sample_id}")
    print(f"Top-20 Neighbors IDs: {similarity_map[sample_id]}")
    print(f"Neighbor SMILES: {combined_df.iloc[similarity_map[sample_id]]['smiles'].tolist()[:3]} ...")

Total molecules loaded: 794386
Computing fingerprints for 794386 molecules...


Computing FPs: 100%|██████████| 80/80 [00:06<00:00, 12.38it/s]


Computed 794386 fingerprints in 7.87 seconds.
Converting fingerprints to packed numpy for Faiss...


Packing FPs: 100%|██████████| 80/80 [00:06<00:00, 12.57it/s]


Packed fingerprints shape: (794386, 256)
Building Faiss IndexBinaryFlat...
Faiss index built with 794386 vectors.
Searching and re-ranking (Parallelized)...


Parallel Search: 100%|██████████| 795/795 [03:48<00:00,  3.48it/s] 


Saved similarity index to similarity_index.json
Sample Molecule ID: 0
Top-20 Neighbors IDs: [568110, 362678, 497923, 55475, 84684, 87682, 177787, 308750, 205272, 242273, 438787, 784068, 695071, 431814, 751567, 111864, 159840, 184334, 401639, 51656]
Neighbor SMILES: ['COc1nc2ccccc2cc1C(=O)CO', 'COc1nc2ccccc2cc1C(=O)CN', 'COc1nc2ccccc2cc1C(=O)NN'] ...


In [7]:
try:
    import faiss
    print("Faiss is available.")
except ImportError:
    print("Faiss is NOT available. Please install it (e.g., pip install faiss-cpu).")

Faiss is available.


In [8]:
# SMILES Tokenization (Character-level)
import json
import concurrent.futures
from tqdm import tqdm

# 1. Collect all unique characters (Parallelized)
all_smiles = combined_df['smiles'].tolist()

def get_unique_chars(smiles_chunk):
    chars = set()
    for s in smiles_chunk:
        chars.update(s)
    return chars

print(f"Collecting unique characters from {len(all_smiles)} SMILES...")

# Split into chunks
chunk_size = 100000
chunks = [all_smiles[i:i + chunk_size] for i in range(0, len(all_smiles), chunk_size)]

unique_chars = set()
# Use ProcessPoolExecutor for CPU-bound task
with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(get_unique_chars, chunks), total=len(chunks), desc="Extracting chars"))

for res in results:
    unique_chars.update(res)

sorted_chars = sorted(list(unique_chars))
print(f"Found {len(sorted_chars)} unique characters: {''.join(sorted_chars)}")

# 2. Build Vocabulary
# Special tokens
specials = ['<pad>', '<start>', '<end>', '<unk>']
vocab = specials + sorted_chars

stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for i, c in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")
print("First 10 tokens:", list(stoi.items())[:10])

# 3. Tokenizer Functions
def tokenize(smiles, max_len=None):
    """
    Converts SMILES string to list of token IDs.
    Adds <start> and <end> tokens.
    Pads to max_len if provided.
    """
    tokens = [stoi['<start>']]
    for c in smiles:
        tokens.append(stoi.get(c, stoi['<unk>']))
    tokens.append(stoi['<end>'])
    
    if max_len is not None:
        if len(tokens) < max_len:
            tokens += [stoi['<pad>']] * (max_len - len(tokens))
        else:
            tokens = tokens[:max_len] # Truncate if too long
            
    return tokens

def detokenize(token_ids):
    """
    Converts list of token IDs back to SMILES string.
    Ignores <pad>, <start>, <end>.
    """
    chars = []
    for tid in token_ids:
        if tid in [stoi['<pad>'], stoi['<start>'], stoi['<end>']]:
            continue
        chars.append(itos.get(tid, ''))
    return ''.join(chars)

# Test Tokenization
if len(all_smiles) > 0:
    test_smiles = all_smiles[0]
    token_ids = tokenize(test_smiles)
    reconstructed = detokenize(token_ids)

    print(f"\nOriginal SMILES: {test_smiles}")
    print(f"Token IDs: {token_ids}")
    print(f"Reconstructed: {reconstructed}")
    print(f"Match: {test_smiles == reconstructed}")

# Check max length distribution to decide on padding
print("Calculating max length...")
lengths = [len(s) for s in all_smiles]
print(f"Max SMILES length: {max(lengths)}")

# Save Vocabulary
vocab_file = 'smiles_vocab.json'
with open(vocab_file, 'w') as f:
    # Save stoi and the ordered vocab list. 
    # itos can be reconstructed from vocab list (index -> item)
    json.dump({'stoi': stoi, 'vocab': vocab}, f)

print(f"Saved vocabulary to {vocab_file}")

Extracting chars: 100%|██████████| 8/8 [00:00<00:00, 46.06it/s]


Found 35 unique characters: #()+-/12345678=@BCFHINOPS[\]clnoprs
Vocabulary size: 39
First 10 tokens: [('<pad>', 0), ('<start>', 1), ('<end>', 2), ('<unk>', 3), ('#', 4), ('(', 5), (')', 6), ('+', 7), ('-', 8), ('/', 9)]

Original SMILES: COc1nc2ccccc2cc1C(=O)O
Token IDs: [1, 21, 26, 32, 10, 34, 32, 11, 32, 32, 32, 32, 32, 11, 32, 32, 10, 21, 5, 18, 26, 6, 26, 2]
Reconstructed: COc1nc2ccccc2cc1C(=O)O
Match: True
Calculating max length...
Max SMILES length: 127
Saved vocabulary to smiles_vocab.json


In [ ]:
# Plot Spectrum for the First Molecule
import matplotlib.pyplot as plt
import numpy as np

# Load the first file fully to access spectral data (previously we only loaded SMILES)
first_file_path = BASE_DIR / 'aligned_chunk_0.parquet'

if first_file_path.exists():
    print(f"Loading {first_file_path} to extract spectra...")
    df_full = pd.read_parquet(first_file_path)
    
    # Get the first molecule (ID 0)
    mol_idx = 0
    mol_data = df_full.iloc[mol_idx]
    
    print(f"Molecule ID: {mol_idx}")
    print(f"SMILES: {mol_data.get('smiles', 'N/A')}")
    print("Available columns:", df_full.columns.tolist())

    # Identify potential spectra columns (usually array-like)
    # We exclude common structural arrays like positions, atomic_numbers, etc.
    exclude_cols = ['pos', 'atomic_numbers', 'force', 'dipole_moment', 'smiles', 'id']
    potential_spectra = []
    
    for col in df_full.columns:
        val = mol_data[col]
        # Check if it's an array/list and has a significant length (likely a spectrum)
        if isinstance(val, (np.ndarray, list)) and len(val) > 10:
             if col not in exclude_cols:
                 potential_spectra.append(col)
    
    if potential_spectra:
        print(f"Found potential spectra columns: {potential_spectra}")
        
        for col in potential_spectra:
            try:
                y_data = mol_data[col]
                
                # Ensure y_data is a flat numpy array of floats
                # The error "setting an array element with a sequence" suggests nested arrays or objects
                y_data = np.array(y_data)
                
                # If it's multidimensional (e.g. shape (N, 1)), flatten it
                if y_data.ndim > 1:
                    y_data = y_data.flatten()
                
                # Ensure it's numeric
                y_data = y_data.astype(float)

                plt.figure(figsize=(10, 4))
                
                # Try to find a corresponding x-axis (frequencies/wavenumbers)
                x_col = None
                # Common names for x-axis
                for candidate in ['frequencies', 'wavenumbers', 'x_axis']:
                    if candidate in df_full.columns:
                        x_col = candidate
                        break
                
                if x_col and len(mol_data[x_col]) == len(y_data):
                    x_data = np.array(mol_data[x_col]).flatten().astype(float)
                    plt.plot(x_data, y_data)
                    plt.xlabel(x_col)
                else:
                    plt.plot(y_data)
                    plt.xlabel("Index")
                    
                plt.ylabel("Intensity")
                plt.title(f"Spectrum: {col} (Molecule ID: {mol_idx})")
                plt.grid(True, alpha=0.3)
                plt.show()
            except Exception as e:
                print(f"Could not plot column '{col}': {e}")
                # Print a sample of the data to debug
                print(f"Sample data from '{col}': {mol_data[col][:5] if len(mol_data[col]) > 0 else 'Empty'}")

    else:
        print("No obvious spectra columns found (arrays > 10 elements).")
        print("Please check the column names printed above.")
else:
    print(f"File {first_file_path} not found.") 

In [9]:
# Check dimensions of specific spectra columns
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('multimodal_spectroscopic_dataset')
first_file_path = BASE_DIR / 'aligned_chunk_0.parquet'

if first_file_path.exists():
    df = pd.read_parquet(first_file_path)
    
    # Define targets based on user query, handling potential naming variations
    targets = ['IR_spectra', 'c_nmr_spectra', 'H_nmr_spectra']
    
    print(f"Checking dimensions for: {targets}")
    
    for target in targets:
        # Find exact or close match in columns
        col_name = None
        if target in df.columns:
            col_name = target
        else:
            # Try case-insensitive or with underscore replacements
            for c in df.columns:
                if c.lower() == target.lower().replace(' ', '_'):
                    col_name = c
                    break
        
        if col_name:
            val = df.iloc[0][col_name]
            if isinstance(val, (np.ndarray, list)):
                val_arr = np.array(val)
                print(f"Column '{col_name}': Shape = {val_arr.shape}")
            else:
                print(f"Column '{col_name}': Not an array, type = {type(val)}")
        else:
            print(f"Column '{target}' not found in dataframe. Available columns: {df.columns.tolist()}")
else:
    print(f"File {first_file_path} not found.")

Checking dimensions for: ['IR_spectra', 'c_nmr_spectra', 'H_nmr_spectra']
Column 'ir_spectra': Shape = (1800,)
Column 'c_nmr_spectra': Shape = (10000,)
Column 'h_nmr_spectra': Shape = (10000,)
